In [11]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [12]:
model_path = "modes/hand_landmarker.task"

In [15]:
# 配置 MediaPipe Hand Landmarker
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode
Image = mp.Image  # 导入 MediaPipe Image 类
ImageFormat = mp.ImageFormat  # 导入 MediaPipe 图像格式

# 创建 Hand Landmarker 实例，配置为视频模式
options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO)

# 打开视频流（例如，使用摄像头）
cap = cv2.VideoCapture(1)

try:
    with HandLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # 转换视频帧为 RGB（因为 OpenCV 默认捕获 BGR 格式）
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # 将 OpenCV 图像转换为 MediaPipe Image 对象
            mp_image = Image(image_format=ImageFormat.SRGB, data=frame_rgb)

            # 检测手部关键点
            results = landmarker.detect(mp_image)

            # 在帧上绘制检测到的手部关键点
            if results:
                for hand_landmarks in results:
                    for landmark in hand_landmarks.landmark:
                        x = int(landmark.x * frame.shape[1])
                        y = int(landmark.y * frame.shape[0])
                        cv2.circle(frame, (x, y), 5, (0, 255, 0), -1)

            # 显示结果
            cv2.imshow('Hand Landmarker', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

except KeyboardInterrupt:
    print("手动中断，正在释放资源...")

finally:
    # 释放资源
    cap.release()
    cv2.destroyAllWindows()

ValueError: Task is not initialized with the image mode. Current running mode:VIDEO